<a href="https://colab.research.google.com/github/seethaladevi2024-cpu/OIBSIP/blob/main/To_Do_App.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Complete To-Do List Web Application for Google Colab
# Features: User Authentication, Task Management, SQLite Database

# Step 1: Install required packages
print("📦 Installing required packages...")
!pip install flask flask-cors werkzeug -q

print("✅ Packages installed successfully!\n")

# Step 2: Import necessary libraries
from flask import Flask, render_template_string, request, redirect, url_for, session, flash
from flask_cors import CORS
from werkzeug.security import generate_password_hash, check_password_hash
import sqlite3
import os
import threading
from datetime import datetime
from google.colab.output import eval_js

# Step 3: Initialize Flask application
app = Flask(__name__)
app.secret_key = 'your-secret-key-change-in-production-12345'  # Required for session management
CORS(app)

# Database configuration
DATABASE = 'todo_app.db'

# Step 4: Database setup functions
def get_db_connection():
    """Create a database connection"""
    conn = sqlite3.connect(DATABASE)
    conn.row_factory = sqlite3.Row  # Access columns by name
    return conn

def init_db():
    """Initialize the database with required tables"""
    conn = get_db_connection()
    cursor = conn.cursor()

    # Create users table
    cursor.execute('''
        CREATE TABLE IF NOT EXISTS users (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            username TEXT UNIQUE NOT NULL,
            password TEXT NOT NULL,
            created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP
        )
    ''')

    # Create tasks table
    cursor.execute('''
        CREATE TABLE IF NOT EXISTS tasks (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            user_id INTEGER NOT NULL,
            title TEXT NOT NULL,
            description TEXT,
            created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP,
            FOREIGN KEY (user_id) REFERENCES users (id)
        )
    ''')

    conn.commit()
    conn.close()
    print("✅ Database initialized successfully!")

# Step 5: HTML Templates

# Login Page Template
LOGIN_TEMPLATE = '''
<!DOCTYPE html>
<html lang="en">
<head>
    <meta charset="UTF-8">
    <meta name="viewport" content="width=device-width, initial-scale=1.0">
    <title>Login - To-Do App</title>
    <style>
        * {
            margin: 0;
            padding: 0;
            box-sizing: border-box;
        }

        body {
            font-family: 'Segoe UI', Tahoma, Geneva, Verdana, sans-serif;
            background: linear-gradient(135deg, #667eea 0%, #764ba2 100%);
            min-height: 100vh;
            display: flex;
            justify-content: center;
            align-items: center;
            padding: 20px;
        }

        .container {
            background: white;
            border-radius: 20px;
            box-shadow: 0 20px 60px rgba(0, 0, 0, 0.3);
            padding: 40px;
            max-width: 400px;
            width: 100%;
        }

        .header {
            text-align: center;
            margin-bottom: 30px;
        }

        .header h1 {
            color: #333;
            font-size: 28px;
            margin-bottom: 5px;
        }

        .header p {
            color: #666;
            font-size: 14px;
        }

        .form-group {
            margin-bottom: 20px;
        }

        label {
            display: block;
            color: #555;
            font-weight: 600;
            margin-bottom: 8px;
            font-size: 14px;
        }

        input[type="text"],
        input[type="password"] {
            width: 100%;
            padding: 12px 15px;
            border: 2px solid #e0e0e0;
            border-radius: 10px;
            font-size: 16px;
            transition: all 0.3s ease;
            background: #f9f9f9;
        }

        input:focus {
            outline: none;
            border-color: #667eea;
            background: white;
        }

        .btn {
            width: 100%;
            padding: 15px;
            background: linear-gradient(135deg, #667eea 0%, #764ba2 100%);
            color: white;
            border: none;
            border-radius: 10px;
            font-size: 18px;
            font-weight: 600;
            cursor: pointer;
            transition: transform 0.2s ease, box-shadow 0.2s ease;
            margin-top: 10px;
        }

        .btn:hover {
            transform: translateY(-2px);
            box-shadow: 0 10px 25px rgba(102, 126, 234, 0.4);
        }

        .btn:active {
            transform: translateY(0);
        }

        .link-text {
            text-align: center;
            margin-top: 20px;
            color: #666;
            font-size: 14px;
        }

        .link-text a {
            color: #667eea;
            text-decoration: none;
            font-weight: 600;
        }

        .link-text a:hover {
            text-decoration: underline;
        }

        .alert {
            padding: 12px 15px;
            border-radius: 8px;
            margin-bottom: 20px;
            font-size: 14px;
        }

        .alert-error {
            background: #fee;
            color: #c33;
            border: 1px solid #fcc;
        }

        .alert-success {
            background: #efe;
            color: #3c3;
            border: 1px solid #cfc;
        }

        .icon {
            font-size: 50px;
            text-align: center;
            margin-bottom: 10px;
        }
    </style>
</head>
<body>
    <div class="container">
        <div class="header">
            <div class="icon">📝</div>
            <h1>Welcome Back</h1>
            <p>Login to access your tasks</p>
        </div>

        {% with messages = get_flashed_messages(with_categories=true) %}
            {% if messages %}
                {% for category, message in messages %}
                    <div class="alert alert-{{ category }}">{{ message }}</div>
                {% endfor %}
            {% endif %}
        {% endwith %}

        <form method="POST" action="{{ url_for('login') }}">
            <div class="form-group">
                <label for="username">Username</label>
                <input type="text" id="username" name="username" required>
            </div>

            <div class="form-group">
                <label for="password">Password</label>
                <input type="password" id="password" name="password" required>
            </div>

            <button type="submit" class="btn">Login</button>
        </form>

        <div class="link-text">
            Don't have an account? <a href="{{ url_for('signup') }}">Sign Up</a>
        </div>
    </div>
</body>
</html>
'''

# Signup Page Template
SIGNUP_TEMPLATE = '''
<!DOCTYPE html>
<html lang="en">
<head>
    <meta charset="UTF-8">
    <meta name="viewport" content="width=device-width, initial-scale=1.0">
    <title>Sign Up - To-Do App</title>
    <style>
        * {
            margin: 0;
            padding: 0;
            box-sizing: border-box;
        }

        body {
            font-family: 'Segoe UI', Tahoma, Geneva, Verdana, sans-serif;
            background: linear-gradient(135deg, #667eea 0%, #764ba2 100%);
            min-height: 100vh;
            display: flex;
            justify-content: center;
            align-items: center;
            padding: 20px;
        }

        .container {
            background: white;
            border-radius: 20px;
            box-shadow: 0 20px 60px rgba(0, 0, 0, 0.3);
            padding: 40px;
            max-width: 400px;
            width: 100%;
        }

        .header {
            text-align: center;
            margin-bottom: 30px;
        }

        .header h1 {
            color: #333;
            font-size: 28px;
            margin-bottom: 5px;
        }

        .header p {
            color: #666;
            font-size: 14px;
        }

        .form-group {
            margin-bottom: 20px;
        }

        label {
            display: block;
            color: #555;
            font-weight: 600;
            margin-bottom: 8px;
            font-size: 14px;
        }

        input[type="text"],
        input[type="password"] {
            width: 100%;
            padding: 12px 15px;
            border: 2px solid #e0e0e0;
            border-radius: 10px;
            font-size: 16px;
            transition: all 0.3s ease;
            background: #f9f9f9;
        }

        input:focus {
            outline: none;
            border-color: #667eea;
            background: white;
        }

        .btn {
            width: 100%;
            padding: 15px;
            background: linear-gradient(135deg, #667eea 0%, #764ba2 100%);
            color: white;
            border: none;
            border-radius: 10px;
            font-size: 18px;
            font-weight: 600;
            cursor: pointer;
            transition: transform 0.2s ease, box-shadow 0.2s ease;
            margin-top: 10px;
        }

        .btn:hover {
            transform: translateY(-2px);
            box-shadow: 0 10px 25px rgba(102, 126, 234, 0.4);
        }

        .btn:active {
            transform: translateY(0);
        }

        .link-text {
            text-align: center;
            margin-top: 20px;
            color: #666;
            font-size: 14px;
        }

        .link-text a {
            color: #667eea;
            text-decoration: none;
            font-weight: 600;
        }

        .link-text a:hover {
            text-decoration: underline;
        }

        .alert {
            padding: 12px 15px;
            border-radius: 8px;
            margin-bottom: 20px;
            font-size: 14px;
        }

        .alert-error {
            background: #fee;
            color: #c33;
            border: 1px solid #fcc;
        }

        .alert-success {
            background: #efe;
            color: #3c3;
            border: 1px solid #cfc;
        }

        .icon {
            font-size: 50px;
            text-align: center;
            margin-bottom: 10px;
        }
    </style>
</head>
<body>
    <div class="container">
        <div class="header">
            <div class="icon">✨</div>
            <h1>Create Account</h1>
            <p>Start organizing your tasks</p>
        </div>

        {% with messages = get_flashed_messages(with_categories=true) %}
            {% if messages %}
                {% for category, message in messages %}
                    <div class="alert alert-{{ category }}">{{ message }}</div>
                {% endfor %}
            {% endif %}
        {% endwith %}

        <form method="POST" action="{{ url_for('signup') }}">
            <div class="form-group">
                <label for="username">Username</label>
                <input type="text" id="username" name="username" required minlength="3">
            </div>

            <div class="form-group">
                <label for="password">Password</label>
                <input type="password" id="password" name="password" required minlength="6">
            </div>

            <div class="form-group">
                <label for="confirm_password">Confirm Password</label>
                <input type="password" id="confirm_password" name="confirm_password" required>
            </div>

            <button type="submit" class="btn">Sign Up</button>
        </form>

        <div class="link-text">
            Already have an account? <a href="{{ url_for('login') }}">Login</a>
        </div>
    </div>
</body>
</html>
'''

# Dashboard Template
DASHBOARD_TEMPLATE = '''
<!DOCTYPE html>
<html lang="en">
<head>
    <meta charset="UTF-8">
    <meta name="viewport" content="width=device-width, initial-scale=1.0">
    <title>Dashboard - To-Do App</title>
    <style>
        * {
            margin: 0;
            padding: 0;
            box-sizing: border-box;
        }

        body {
            font-family: 'Segoe UI', Tahoma, Geneva, Verdana, sans-serif;
            background: linear-gradient(135deg, #667eea 0%, #764ba2 100%);
            min-height: 100vh;
            padding: 20px;
        }

        .navbar {
            background: white;
            border-radius: 15px;
            padding: 20px 30px;
            margin-bottom: 20px;
            display: flex;
            justify-content: space-between;
            align-items: center;
            box-shadow: 0 5px 15px rgba(0, 0, 0, 0.1);
        }

        .navbar h1 {
            color: #333;
            font-size: 24px;
        }

        .navbar .user-info {
            display: flex;
            align-items: center;
            gap: 15px;
        }

        .navbar .username {
            color: #666;
            font-weight: 600;
        }

        .btn-logout {
            padding: 10px 20px;
            background: #e74c3c;
            color: white;
            border: none;
            border-radius: 8px;
            font-size: 14px;
            font-weight: 600;
            cursor: pointer;
            text-decoration: none;
            display: inline-block;
        }

        .btn-logout:hover {
            background: #c0392b;
        }

        .container {
            max-width: 1200px;
            margin: 0 auto;
        }

        .add-task-section {
            background: white;
            border-radius: 15px;
            padding: 30px;
            margin-bottom: 20px;
            box-shadow: 0 5px 15px rgba(0, 0, 0, 0.1);
        }

        .add-task-section h2 {
            color: #333;
            margin-bottom: 20px;
        }

        .form-group {
            margin-bottom: 15px;
        }

        label {
            display: block;
            color: #555;
            font-weight: 600;
            margin-bottom: 8px;
        }

        input[type="text"],
        textarea {
            width: 100%;
            padding: 12px 15px;
            border: 2px solid #e0e0e0;
            border-radius: 10px;
            font-size: 16px;
            font-family: inherit;
            background: #f9f9f9;
            transition: all 0.3s ease;
        }

        input:focus,
        textarea:focus {
            outline: none;
            border-color: #667eea;
            background: white;
        }

        textarea {
            resize: vertical;
            min-height: 100px;
        }

        .btn-add {
            padding: 12px 30px;
            background: linear-gradient(135deg, #667eea 0%, #764ba2 100%);
            color: white;
            border: none;
            border-radius: 10px;
            font-size: 16px;
            font-weight: 600;
            cursor: pointer;
            transition: transform 0.2s ease;
        }

        .btn-add:hover {
            transform: translateY(-2px);
        }

        .tasks-section {
            background: white;
            border-radius: 15px;
            padding: 30px;
            box-shadow: 0 5px 15px rgba(0, 0, 0, 0.1);
        }

        .tasks-section h2 {
            color: #333;
            margin-bottom: 20px;
        }

        .task-item {
            background: #f9f9f9;
            border-left: 4px solid #667eea;
            padding: 20px;
            margin-bottom: 15px;
            border-radius: 10px;
            display: flex;
            justify-content: space-between;
            align-items: start;
            transition: all 0.3s ease;
        }

        .task-item:hover {
            box-shadow: 0 5px 15px rgba(0, 0, 0, 0.1);
            transform: translateX(5px);
        }

        .task-content {
            flex: 1;
        }

        .task-title {
            color: #333;
            font-size: 18px;
            font-weight: 600;
            margin-bottom: 8px;
        }

        .task-description {
            color: #666;
            font-size: 14px;
            line-height: 1.5;
            margin-bottom: 10px;
        }

        .task-date {
            color: #999;
            font-size: 12px;
        }

        .btn-delete {
            padding: 8px 15px;
            background: #e74c3c;
            color: white;
            border: none;
            border-radius: 8px;
            font-size: 14px;
            cursor: pointer;
            transition: background 0.3s ease;
        }

        .btn-delete:hover {
            background: #c0392b;
        }

        .no-tasks {
            text-align: center;
            color: #999;
            padding: 40px;
            font-size: 16px;
        }

        .alert {
            padding: 12px 15px;
            border-radius: 8px;
            margin-bottom: 20px;
            font-size: 14px;
        }

        .alert-error {
            background: #fee;
            color: #c33;
            border: 1px solid #fcc;
        }

        .alert-success {
            background: #efe;
            color: #3c3;
            border: 1px solid #cfc;
        }
    </style>
</head>
<body>
    <div class="navbar">
        <h1>📝 My To-Do Dashboard</h1>
        <div class="user-info">
            <span class="username">👤 {{ username }}</span>
            <a href="{{ url_for('logout') }}" class="btn-logout">Logout</a>
        </div>
    </div>

    <div class="container">
        {% with messages = get_flashed_messages(with_categories=true) %}
            {% if messages %}
                {% for category, message in messages %}
                    <div class="alert alert-{{ category }}">{{ message }}</div>
                {% endfor %}
            {% endif %}
        {% endwith %}

        <div class="add-task-section">
            <h2>➕ Add New Task</h2>
            <form method="POST" action="{{ url_for('add_task') }}">
                <div class="form-group">
                    <label for="title">Task Title</label>
                    <input type="text" id="title" name="title" required placeholder="E.g., Buy groceries">
                </div>

                <div class="form-group">
                    <label for="description">Description (Optional)</label>
                    <textarea id="description" name="description" placeholder="Add more details about your task..."></textarea>
                </div>

                <button type="submit" class="btn-add">Add Task</button>
            </form>
        </div>

        <div class="tasks-section">
            <h2>📋 Your Tasks ({{ tasks|length }})</h2>

            {% if tasks %}
                {% for task in tasks %}
                <div class="task-item">
                    <div class="task-content">
                        <div class="task-title">{{ task.title }}</div>
                        {% if task.description %}
                        <div class="task-description">{{ task.description }}</div>
                        {% endif %}
                        <div class="task-date">Created: {{ task.created_at }}</div>
                    </div>
                    <form method="POST" action="{{ url_for('delete_task', task_id=task.id) }}" style="display: inline;">
                        <button type="submit" class="btn-delete" onclick="return confirm('Are you sure you want to delete this task?')">Delete</button>
                    </form>
                </div>
                {% endfor %}
            {% else %}
                <div class="no-tasks">
                    <p>🎉 No tasks yet! Add your first task above.</p>
                </div>
            {% endif %}
        </div>
    </div>
</body>
</html>
'''

# Step 6: Define routes

@app.route('/')
def index():
    """Redirect to login if not authenticated, otherwise to dashboard"""
    if 'user_id' in session:
        return redirect(url_for('dashboard'))
    return redirect(url_for('login'))

@app.route('/login', methods=['GET', 'POST'])
def login():
    """Handle user login"""
    if request.method == 'POST':
        username = request.form.get('username')
        password = request.form.get('password')

        conn = get_db_connection()
        user = conn.execute('SELECT * FROM users WHERE username = ?', (username,)).fetchone()
        conn.close()

        if user and check_password_hash(user['password'], password):
            # Login successful
            session['user_id'] = user['id']
            session['username'] = user['username']
            flash('Login successful! Welcome back.', 'success')
            return redirect(url_for('dashboard'))
        else:
            flash('Invalid username or password. Please try again.', 'error')

    return render_template_string(LOGIN_TEMPLATE)

@app.route('/signup', methods=['GET', 'POST'])
def signup():
    """Handle user registration"""
    if request.method == 'POST':
        username = request.form.get('username')
        password = request.form.get('password')
        confirm_password = request.form.get('confirm_password')

        # Validation
        if len(username) < 3:
            flash('Username must be at least 3 characters long.', 'error')
            return render_template_string(SIGNUP_TEMPLATE)

        if len(password) < 6:
            flash('Password must be at least 6 characters long.', 'error')
            return render_template_string(SIGNUP_TEMPLATE)

        if password != confirm_password:
            flash('Passwords do not match!', 'error')
            return render_template_string(SIGNUP_TEMPLATE)

        # Hash the password
        hashed_password = generate_password_hash(password)

        # Try to insert user
        try:
            conn = get_db_connection()
            conn.execute('INSERT INTO users (username, password) VALUES (?, ?)',
                        (username, hashed_password))
            conn.commit()
            conn.close()
            flash('Account created successfully! Please login.', 'success')
            return redirect(url_for('login'))
        except sqlite3.IntegrityError:
            flash('Username already exists. Please choose a different one.', 'error')

    return render_template_string(SIGNUP_TEMPLATE)

@app.route('/dashboard')
def dashboard():
    """Display user's dashboard with tasks"""
    if 'user_id' not in session:
        flash('Please login to access the dashboard.', 'error')
        return redirect(url_for('login'))

    # Get user's tasks
    conn = get_db_connection()
    tasks = conn.execute('SELECT * FROM tasks WHERE user_id = ? ORDER BY created_at DESC',
                        (session['user_id'],)).fetchall()
    conn.close()

    return render_template_string(DASHBOARD_TEMPLATE,
                                 username=session['username'],
                                 tasks=tasks)

@app.route('/add_task', methods=['POST'])
def add_task():
    """Add a new task"""
    if 'user_id' not in session:
        return redirect(url_for('login'))

    title = request.form.get('title')
    description = request.form.get('description')

    if not title:
        flash('Task title is required!', 'error')
        return redirect(url_for('dashboard'))

    # Insert task
    conn = get_db_connection()
    conn.execute('INSERT INTO tasks (user_id, title, description) VALUES (?, ?, ?)',
                (session['user_id'], title, description))
    conn.commit()
    conn.close()

    flash('Task added successfully!', 'success')
    return redirect(url_for('dashboard'))

@app.route('/delete_task/<int:task_id>', methods=['POST'])
def delete_task(task_id):
    """Delete a task"""
    if 'user_id' not in session:
        return redirect(url_for('login'))

    # Delete only if task belongs to current user
    conn = get_db_connection()
    conn.execute('DELETE FROM tasks WHERE id = ? AND user_id = ?',
                (task_id, session['user_id']))
    conn.commit()
    conn.close()

    flash('Task deleted successfully!', 'success')
    return redirect(url_for('dashboard'))

@app.route('/logout')
def logout():
    """Handle user logout"""
    session.clear()
    flash('You have been logged out successfully.', 'success')
    return redirect(url_for('login'))

# Step 7: Initialize database and run the application
def main():
    """Main function to initialize and run the app"""
    print("🗄️  Initializing database...")
    init_db()

    print("\n🚀 Starting Flask server...")
    # Start Flask in a background thread
    threading.Thread(
        target=lambda: app.run(host='0.0.0.0', port=5000, debug=False, use_reloader=False),
        daemon=True
    ).start()

    # Wait for Flask to start
    import time
    time.sleep(3)

    # Get public URL from Colab
    print("🌐 Getting public URL from Google Colab...")
    try:
        public_url = eval_js("google.colab.kernel.proxyPort(5000)")
        print(f"\n" + "="*60)
        print(f"✅ SUCCESS! Your To-Do App is running!")
        print(f"="*60)
        print(f"\n🔗 Public URL: {public_url}")
        print(f"\n📱 Click the link above to access your application!")
        print(f"\n⚠️  Keep this cell running to maintain the connection.")
        print(f"\n📝 Features:")
        print(f"   • User Signup & Login")
        print(f"   • Add, View, Delete Tasks")
        print(f"   • Secure Password Storage")
        print(f"   • User-specific Data")
        print(f"\n💡 Tip: Create an account to get started!")
        print(f"="*60)
    except Exception as e:
        print(f"\n⚠️  Could not automatically get public URL.")
        print(f"📋 The Flask server is running on port 5000")
        print(f"   Look for the 'Open in new tab' icon in the cell output")

# Run the application
main()

📦 Installing required packages...
✅ Packages installed successfully!

🗄️  Initializing database...
✅ Database initialized successfully!

🚀 Starting Flask server...
 * Serving Flask app '__main__'
 * Debug mode: off


Address already in use
Port 5000 is in use by another program. Either identify and stop that program, or start the server with a different port.


🌐 Getting public URL from Google Colab...

✅ SUCCESS! Your To-Do App is running!

🔗 Public URL: https://5000-m-s-azd49hqqo6pk-c.us-west1-2.prod.colab.dev

📱 Click the link above to access your application!

⚠️  Keep this cell running to maintain the connection.

📝 Features:
   • User Signup & Login
   • Add, View, Delete Tasks
   • Secure Password Storage
   • User-specific Data

💡 Tip: Create an account to get started!
